# LeetCode 1666 — Change the Root of a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/change-the-root-of-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree · **Premium**

> **Ghi chú (bài Premium):** đề bài trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: doocs/leetcode). Nếu có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None
        self.parent = None
"""

class Solution:
    def flipBinaryTree(self, root: 'Node', leaf: 'Node') -> 'Node':
        ...
```

## 1. Mục tiêu học

1. Làm quen với cây nhị phân có **con trỏ `parent`** và hiểu vì sao phải cập nhật **cả ba** con trỏ `left`, `right`, `parent` khi thao tác cây.
2. Mô phỏng chính xác quy tắc "đổi gốc" của đề trên **đường đi từ lá lên gốc**, và hiểu vì sao mỗi nút trên đường đi chỉ còn **nhiều nhất một con** khi tới lượt nó.
3. Viết ba lời giải: (1) tìm đường đi bằng DFS + dựng lại `parent` bằng BFS, (2) đi ngược bằng `parent` tại chỗ `O(h)`, (3) đệ quy.
4. Tự viết **hàm kiểm tra hợp lệ** (validator) độc lập: tập cạnh không đổi, lá thành gốc, `parent` khớp với `left/right`, vị trí con đúng quy tắc.
5. Dựng/xuất cây kiểu LeetCode (level-order có `null`) để chạy thử mọi ví dụ.

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân và một nút **lá** `leaf`. Hãy **đổi gốc** cây sao cho `leaf` trở thành gốc mới.

Cách đổi gốc: với mỗi nút `cur` trên đường đi **bắt đầu từ `leaf` đi lên tới `root`, không tính `root`**, thực hiện:
1. Nếu `cur` có con trái, con đó trở thành **con phải** của `cur`.
2. Cha ban đầu của `cur` trở thành **con trái** của `cur`. Lưu ý: trong quá trình này, con trỏ từ cha ban đầu xuống `cur` bị đặt thành `null`, nên cha ban đầu còn **nhiều nhất một con**.

Trả về gốc mới của cây sau khi đổi gốc.

**Lưu ý:** phải đặt đúng các con trỏ `Node.parent` sau khi đổi gốc, nếu không sẽ bị "Wrong Answer".

**Ví dụ 1:** `root = [3,5,1,6,2,0,8,null,null,7,4]`, `leaf = 7` → `[7,2,null,5,4,3,6,null,null,null,1,null,null,0,8]`

**Ví dụ 2:** `root = [3,5,1,6,2,0,8,null,null,7,4]`, `leaf = 0` → `[0,1,null,3,8,5,null,null,null,6,2,null,null,7,4]`

**Ràng buộc:**
- Số nút trong `[2, 100]`.
- `-10⁹ ≤ Node.val ≤ 10⁹`, mọi giá trị **khác nhau**.
- `leaf` tồn tại trong cây (và là lá).

Trên LeetCode, input `leaf = 7` nghĩa là **nút có giá trị 7** — ta tự tìm nút đó sau khi dựng cây.

In [1]:
import random
import time
from collections import deque
from typing import Optional, List

class Node:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None
        self.parent = None
    def __repr__(self):
        return f"Node({self.val})"

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = null), có gán parent."""
    if not arr or arr[0] is None:
        return None
    root = Node(arr[0])
    q = deque([root]); i = 1
    while q and i < len(arr):
        x = q.popleft()
        if i < len(arr) and arr[i] is not None:
            x.left = Node(arr[i]); x.left.parent = x; q.append(x.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            x.right = Node(arr[i]); x.right.parent = x; q.append(x.right)
        i += 1
    return root

def serialize(root):
    """Xuất cây ra level-order kiểu LeetCode (cắt các None ở cuối)."""
    out, q = [], deque([root])
    while q:
        x = q.popleft()
        if x is None:
            out.append(None); continue
        out.append(x.val); q.append(x.left); q.append(x.right)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    st = [root]
    while st:
        x = st.pop()
        if x.val == val:
            return x
        if x.left: st.append(x.left)
        if x.right: st.append(x.right)
    return None

EX_TREE = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
examples = [
    (EX_TREE, 7, [7, 2, None, 5, 4, 3, 6, None, None, None, 1, None, None, 0, 8]),
    (EX_TREE, 0, [0, 1, None, 3, 8, 5, None, None, None, 6, 2, None, None, 7, 4]),
]
root = build_tree(EX_TREE)
print("cây ví dụ:", serialize(root))
print("nút 7:", find(root, 7), "cha:", find(root, 7).parent, "ông:", find(root, 7).parent.parent)

cây ví dụ: [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
nút 7: Node(7) cha: Node(2) ông: Node(5)


## 3. Kiến thức nền

### 3.1 Cây nhị phân có con trỏ `parent`

Mỗi nút có `left`, `right`, `parent`. Ràng buộc nhất quán: nếu `x.left is c` thì `c.parent is x` (tương tự với `right`), và gốc có `parent = None`. Hàm dưới đây kiểm tra điều đó — ta sẽ dùng nó trong mọi test.

In [2]:
def parents_ok(root):
    """True nếu mọi con trỏ parent khớp với left/right và gốc có parent None."""
    if root.parent is not None:
        return False
    st = [root]
    seen = set()
    while st:
        x = st.pop()
        if id(x) in seen:          # chu trình -> cây hỏng
            return False
        seen.add(id(x))
        for c in (x.left, x.right):
            if c is not None:
                if c.parent is not x:
                    return False
                st.append(c)
    return True

print("cây ví dụ hợp lệ parent:", parents_ok(build_tree(EX_TREE)))
r = build_tree([1, 2, 3]); r.left.parent = r.right     # cố tình làm hỏng
print("cây bị làm hỏng:", parents_ok(r))

cây ví dụ hợp lệ parent: True
cây bị làm hỏng: False


### 3.2 Đi từ một nút lên gốc bằng `parent`

Lặp `x = x.parent` cho tới khi `x` là `None` — được đường đi **lá → gốc** trong `O(h)` (`h` là chiều cao).

In [3]:
def path_up(node):
    path = []
    while node is not None:
        path.append(node.val)
        node = node.parent
    return path

root = build_tree(EX_TREE)
print("đường đi 7 -> gốc:", path_up(find(root, 7)))
print("đường đi 0 -> gốc:", path_up(find(root, 0)))

đường đi 7 -> gốc: [7, 2, 5, 3]
đường đi 0 -> gốc: [0, 1, 3]


### 3.3 Tìm đường đi gốc → nút bằng DFS (không dùng `parent`)

Khi không tin con trỏ `parent` (hoặc cây không có), dùng DFS với stack chứa `(nút, đường đi tới nút)`.

In [4]:
def path_down(root, target):
    st = [(root, [root])]
    while st:
        x, p = st.pop()
        if x is target:
            return p
        for c in (x.right, x.left):
            if c is not None:
                st.append((c, p + [c]))
    return None

root = build_tree(EX_TREE)
print("gốc -> 7:", [x.val for x in path_down(root, find(root, 7))])

gốc -> 7: [3, 5, 2, 7]


### 3.4 So sánh định danh `is` và gán nhiều con trỏ

`p.left is cur` so sánh **cùng một đối tượng** (không phải cùng giá trị). Khi đổi con trỏ, thứ tự rất quan trọng: phải **lưu lại** `p.parent` (ông) **trước** khi ghi đè bất cứ thứ gì, nếu không sẽ mất đường đi lên.

In [5]:
a, b = Node(1), Node(1)
print("a == b ?", a == b, "| a is a ?", a is a)   # Node không định nghĩa __eq__ -> == cũng so định danh
x = build_tree([1, 2])
cur, p = x.left, x
gp = p.parent            # lưu ông TRƯỚC (ở đây là None)
cur.left, p.left = p, None
p.parent, cur.parent = cur, gp
print("sau khi lật cạnh 1-2:", serialize(cur), "| parent ok:", parents_ok(cur))

a == b ? False | a is a ? True
sau khi lật cạnh 1-2: [2, 1] | parent ok: True


## 4. Ý tưởng nền tảng / trực giác — tính bằng tay

Cây ví dụ:

```
            3
          /   \
         5     1
        / \   / \
       6   2 0   8
          / \
         7   4
```

Đổi gốc về `leaf = 7`. Đường đi lá → gốc: `7 → 2 → 5 → 3`. Xử lý lần lượt `cur = 7, 2, 5` (không xử lý gốc `3`):

| cur | con trái hiện tại (sau khi đã tách con trên đường đi) | bước 1: trái → phải | bước 2: cha thành con trái | cha bị cắt cạnh xuống cur |
|---|---|---|---|---|
| 7 | không có | — | `7.left = 2` | `2.left = null` |
| 2 | không có (7 đã bị cắt) — còn con phải 4 | — | `2.left = 5` | `5.right = null` |
| 5 | `6` | `5.right = 6` | `5.left = 3` | `3.left = null` |
| 3 (gốc) | dừng | | | |

Kết quả:

```
        7
       /
      2
     / \
    5   4
   / \
  3   6
   \
    1
   / \
  0   8
```

Level-order: `[7,2,null,5,4,3,6,null,null,null,1,null,null,0,8]` — khớp ví dụ 1.

**Quan sát then chốt:** khi tới lượt `cur`, con trên đường đi (nút đã xử lý trước) **đã bị cắt** khỏi `cur`, nên `cur` có tối đa **một** con còn lại. Nếu con đó ở bên trái thì chuyển sang phải, để chỗ trái cho cha cũ. Nhờ vậy **không bao giờ** đè mất một con nào.

In [6]:
def print_tree(root, indent=0, tag="root"):
    """In cây dạng thụt lề: mỗi dòng một nút, kèm parent để kiểm tra."""
    if root is None:
        return
    par = root.parent.val if root.parent else None
    print("    " * indent + f"{tag}: {root.val} (parent={par})")
    print_tree(root.left, indent + 1, "L")
    print_tree(root.right, indent + 1, "R")

print_tree(build_tree(EX_TREE))

root: 3 (parent=None)
    L: 5 (parent=3)
        L: 6 (parent=5)
        R: 2 (parent=5)
            L: 7 (parent=2)
            R: 4 (parent=2)
    R: 1 (parent=3)
        L: 0 (parent=1)
        R: 8 (parent=1)


## 5. Cách 1 — Baseline: tìm đường đi bằng DFS, làm theo quy tắc, rồi dựng lại toàn bộ `parent`

**Ý tưởng:** cách "an toàn" nhất cho người mới — không phụ thuộc vào việc cập nhật `parent` từng bước:
1. Tìm đường đi `gốc → leaf` bằng DFS (mục 3.3), đảo lại thành `leaf → gốc`.
2. Với mỗi `cur = path[k]` (trừ gốc), cha cũ là `p = path[k+1]`: cắt `p` khỏi con của nó (con trỏ xuống `cur`), nếu `cur.left` còn thì chuyển sang phải, rồi `cur.left = p`.
   Thứ tự xử lý từ lá lên đảm bảo khi tới `cur`, con trên đường đi đã bị cắt.
3. Duyệt BFS từ gốc mới, **gán lại mọi `parent`**.

Chi phí: DFS `O(n)` (lưu đường đi theo kiểu sao chép list nên tệ nhất `O(n·h)`), BFS `O(n)`.

In [7]:
def flip_baseline(root, leaf):
    path = path_down(root, leaf)[::-1]          # leaf -> ... -> root
    for k in range(len(path) - 1):
        cur, p = path[k], path[k + 1]
        if p.left is cur:                        # cha cắt cạnh xuống cur
            p.left = None
        else:
            p.right = None
        if cur.left is not None:                 # bước 1
            cur.right = cur.left
        cur.left = p                             # bước 2
    new_root = leaf
    new_root.parent = None                       # dựng lại toàn bộ parent bằng BFS
    q = deque([new_root])
    while q:
        x = q.popleft()
        for c in (x.left, x.right):
            if c is not None:
                c.parent = x
                q.append(c)
    return new_root

for arr, lv, expected in examples:
    root = build_tree(arr)
    nr = flip_baseline(root, find(root, lv))
    print("leaf", lv, "->", serialize(nr), "| parent ok:", parents_ok(nr))
    assert serialize(nr) == expected and parents_ok(nr)

leaf 7 -> [7, 2, None, 5, 4, 3, 6, None, None, None, 1, None, None, 0, 8] | parent ok: True
leaf 0 -> [0, 1, None, 3, 8, 5, None, None, None, 6, 2, None, None, 7, 4] | parent ok: True


**Độ phức tạp Cách 1:** Time `O(n·h)` do sao chép đường đi trong DFS (có thể giảm về `O(n)` bằng dict cha), Space `O(n)`.

**Nút thắt:** ta duyệt **toàn bộ cây** (DFS tìm lá + BFS gán parent) trong khi chỉ các nút trên đường đi thay đổi. Đề đã cho sẵn con trỏ `parent` — nên dùng nó để đi thẳng từ lá lên, chỉ chạm `O(h)` nút.

Để thấy rõ tốn bao nhiêu nút được "chạm" ở mỗi cách, đếm trên một cây ngẫu nhiên 100 nút:

In [8]:
def random_tree(n, rnd):
    """Cây nhị phân ngẫu nhiên n nút, giá trị khác nhau, có parent."""
    vals = rnd.sample(range(-10**9, 10**9), n)
    nodes = [Node(v) for v in vals]
    root = nodes[0]
    free = [(root, "L"), (root, "R")]
    for x in nodes[1:]:
        k = rnd.randrange(len(free))
        p, side = free[k]
        free[k] = free[-1]; free.pop()
        if side == "L": p.left = x
        else: p.right = x
        x.parent = p
        free.append((x, "L")); free.append((x, "R"))
    return root

def leaves(root):
    res, st = [], [root]
    while st:
        x = st.pop()
        if x.left is None and x.right is None:
            res.append(x)
        for c in (x.left, x.right):
            if c: st.append(c)
    return res

rnd = random.Random(1)
t = random_tree(100, rnd)
lf = leaves(t)[0]
print("số nút n = 100, độ sâu của lá được chọn h =", len(path_up(lf)) - 1)
print("Cách 1 chạm ~ toàn bộ 100 nút (DFS + BFS); cách dùng parent chỉ chạm", len(path_up(lf)), "nút")

số nút n = 100, độ sâu của lá được chọn h = 1
Cách 1 chạm ~ toàn bộ 100 nút (DFS + BFS); cách dùng parent chỉ chạm 2 nút


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** quy tắc của đề thực chất là **đảo chiều mọi cạnh trên đường đi lá → gốc**. Mỗi nút `cur` trên đường đi:
- mất con "đi xuống phía lá" (vì nút đó giờ là **cha** của nó),
- nhận cha cũ làm **con trái**,
- con còn lại (nếu có) nằm ở **bên phải**.

Đi từ lá lên bằng `parent`, mỗi bước cần **4 con trỏ**: `cur`, `p = cur.parent`, `gp = p.parent` (lưu trước!) và con còn lại của `cur`.

**Quy tắc "nếu… thì…" cho mỗi bước (`cur ≠ root`):**
- Nếu `cur.left` khác `None` **thì** `cur.right = cur.left`.
- `cur.left = p`; `p.parent = cur`.
- Nếu `p.left is cur` **thì** `p.left = None`, ngược lại `p.right = None`.
- Tiến lên: `cur = p`, `p = gp`.
- Kết thúc: `leaf.parent = None`.

**Invariant:** trước khi xử lý `cur`, mọi nút từ `leaf` tới con của `cur` trên đường đi đã được lật xong; `cur` không còn trỏ xuống nút trên đường đi (đã bị cắt ở bước trước), và `p`, `gp` vẫn là cha/ông **cũ** của `cur`.

Một chi tiết tinh tế: ở bước 1 (`cur.right = cur.left`), có thể `cur` vốn có con phải không? Nếu `cur` không phải lá thì nó có con trên đường đi (đã bị cắt) và **tối đa** một con khác. Nếu con khác nằm bên phải thì `cur.left` lúc này là `None` (con trên đường đi là con trái và đã bị cắt) ⇒ không ghi đè. Nếu con khác nằm bên trái thì `cur.right` là `None` (con trên đường đi là con phải, đã cắt) ⇒ ghi an toàn.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đi ngược bằng `parent`, lật tại chỗ (`O(h)`)

**Từng đoạn:**
1. `cur = leaf`, `p = leaf.parent`.
2. Lặp khi `cur is not root`: lưu `gp = p.parent`; chuyển con trái của `cur` sang phải (nếu có); `cur.left = p`, `p.parent = cur`; cắt con trỏ từ `p` xuống `cur`; `cur, p = p, gp`.
3. `leaf.parent = None`, trả `leaf`.

Code demo — in con trỏ sau mỗi bước:

In [9]:
def demo_iterative(root, leaf):
    cur, p = leaf, leaf.parent
    step = 0
    while cur is not root:
        gp = p.parent
        if cur.left is not None:
            cur.right = cur.left
        cur.left = p
        p.parent = cur
        if p.left is cur:
            p.left = None
        else:
            p.right = None
        step += 1
        print(f"bước {step}: cur={cur.val}: left={cur.left.val}, right={cur.right.val if cur.right else None};"
              f" cha cũ {p.val} giờ có left={p.left.val if p.left else None}, right={p.right.val if p.right else None}")
        cur, p = p, gp
    leaf.parent = None
    return leaf

root = build_tree(EX_TREE)
nr = demo_iterative(root, find(root, 7))
print("kết quả:", serialize(nr))

bước 1: cur=7: left=2, right=None; cha cũ 2 giờ có left=None, right=4
bước 2: cur=2: left=5, right=4; cha cũ 5 giờ có left=6, right=None
bước 3: cur=5: left=3, right=6; cha cũ 3 giờ có left=None, right=1
kết quả: [7, 2, None, 5, 4, 3, 6, None, None, None, 1, None, None, 0, 8]


Code nộp LeetCode của Cách 2:

In [10]:
class SolutionIter:
    def flipBinaryTree(self, root: 'Node', leaf: 'Node') -> 'Node':
        cur, p = leaf, leaf.parent
        while cur is not root:
            gp = p.parent                 # lưu ông trước khi ghi đè
            if cur.left is not None:      # bước 1: trái -> phải
                cur.right = cur.left
            cur.left = p                  # bước 2: cha cũ thành con trái
            p.parent = cur
            if p.left is cur:             # cha cũ cắt cạnh xuống cur
                p.left = None
            else:
                p.right = None
            cur, p = p, gp
        leaf.parent = None
        return leaf

for arr, lv, expected in examples:
    root = build_tree(arr)
    nr = SolutionIter().flipBinaryTree(root, find(root, lv))
    assert serialize(nr) == expected and parents_ok(nr)
print("Cách 2 qua 2 ví dụ")

Cách 2 qua 2 ví dụ


**Độ phức tạp Cách 2:** Time `O(h)` (chỉ các nút trên đường đi), Space `O(1)`.

Chú ý: `cur.right` sau khi xử lý **không** bao giờ trỏ lại `cur` cũ nhờ cắt cạnh ở cha; còn `cur.left` cũ (nếu có) đã được chuyển sang phải trước khi ta ghi `cur.left = p`.

### 7.2 Cách 3 — Đệ quy "lật cạnh từ trên xuống"

Tư duy khác: định nghĩa `attach(cur, new_parent)` = "biến `cur` thành con trái của `new_parent`, và lật đệ quy phần phía trên `cur`". Quá trình:
- Lưu `p = cur.parent` (cha cũ). Đặt `cur.parent = new_parent`.
- Nếu `cur is root` thì xong (gốc cũ chỉ cần đổi parent và đã bị cắt con ở bước trước).
- Ngược lại: cắt `p` khỏi `cur` ở phía `p`, chuyển con trái của `cur` sang phải nếu có, gán `cur.left = p`, gọi đệ quy `attach(p, cur)`.

Gọi `attach(leaf, None)`. Độ sâu đệ quy = `h ≤ 99` (n ≤ 100) — an toàn với giới hạn đệ quy của Python.

Code demo:

In [11]:
def flip_recursive(root, leaf):
    def attach(cur, new_parent, depth=0):
        p = cur.parent
        cur.parent = new_parent
        print("  " * depth + f"attach({cur.val}, parent mới={new_parent.val if new_parent else None})")
        if cur is root:
            return
        if p.left is cur:
            p.left = None
        else:
            p.right = None
        if cur.left is not None:
            cur.right = cur.left
        cur.left = p
        attach(p, cur, depth + 1)
    attach(leaf, None)
    return leaf

root = build_tree(EX_TREE)
nr = flip_recursive(root, find(root, 0))
print("leaf 0 ->", serialize(nr))

attach(0, parent mới=None)
  attach(1, parent mới=0)
    attach(3, parent mới=1)
leaf 0 -> [0, 1, None, 3, 8, 5, None, None, None, 6, 2, None, None, 7, 4]


Code nộp LeetCode của Cách 3 (không in):

In [12]:
class SolutionRec:
    def flipBinaryTree(self, root: 'Node', leaf: 'Node') -> 'Node':
        def attach(cur, new_parent):
            p = cur.parent
            cur.parent = new_parent
            if cur is root:
                return
            if p.left is cur:
                p.left = None
            else:
                p.right = None
            if cur.left is not None:
                cur.right = cur.left
            cur.left = p
            attach(p, cur)
        attach(leaf, None)
        return leaf

for arr, lv, expected in examples:
    root = build_tree(arr)
    nr = SolutionRec().flipBinaryTree(root, find(root, lv))
    assert serialize(nr) == expected and parents_ok(nr)
print("Cách 3 qua 2 ví dụ")

Cách 3 qua 2 ví dụ


**Độ phức tạp Cách 3:** Time `O(h)`, Space `O(h)` cho ngăn xếp đệ quy.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 (`leaf = 0`). Đường đi: `0 → 1 → 3`.

| bước | cur | p | gp | con trái của cur trước bước 1 | sau bước: cur.left / cur.right | p sau khi cắt (left, right) |
|---|---|---|---|---|---|---|
| 1 | 0 | 1 | 3 | — | 1 / — | (—, 8) |
| 2 | 1 | 3 | None | — (0 đã cắt) | 3 / 8 | (5, —) |
| dừng | 3 = root | | | | | |

Kết quả: gốc `0`, `0.left = 1`, `1.left = 3`, `1.right = 8`, `3.left = 5` (cây con 5 giữ nguyên) → `[0,1,null,3,8,5,null,null,null,6,2,null,null,7,4]`.

Hàm `trace_flip` in trạng thái mỗi vòng:

In [13]:
def trace_flip(root, leaf):
    def v(x): return x.val if x is not None else None
    cur, p = leaf, leaf.parent
    print(" bước | cur |  p  | gp   | cur.left trước | cur.left/right sau | p.left/right sau")
    k = 0
    while cur is not root:
        gp = p.parent
        before = v(cur.left)
        if cur.left is not None:
            cur.right = cur.left
        cur.left = p
        p.parent = cur
        if p.left is cur:
            p.left = None
        else:
            p.right = None
        k += 1
        print(f" {k:4} | {cur.val:3} | {p.val:3} | {str(v(gp)):4} | {str(before):14} | {v(cur.left)!s:>5}/{v(cur.right)!s:<12} | {v(p.left)}/{v(p.right)}")
        cur, p = p, gp
    leaf.parent = None
    return leaf

root = build_tree(EX_TREE)
nr = trace_flip(root, find(root, 0))
print("serialize:", serialize(nr))
print_tree(nr)

 bước | cur |  p  | gp   | cur.left trước | cur.left/right sau | p.left/right sau
    1 |   0 |   1 | 3    | None           |     1/None         | None/8
    2 |   1 |   3 | None | None           |     3/8            | 5/None
serialize: [0, 1, None, 3, 8, 5, None, None, None, 6, 2, None, None, 7, 4]
root: 0 (parent=None)
    L: 1 (parent=0)
        L: 3 (parent=1)
            L: 5 (parent=3)
                L: 6 (parent=5)
                R: 2 (parent=5)
                    L: 7 (parent=2)
                    R: 4 (parent=2)
        R: 8 (parent=1)


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def flipBinaryTree(self, root: 'Node', leaf: 'Node') -> 'Node':
        cur, p = leaf, leaf.parent
        while cur is not root:
            gp = p.parent
            if cur.left:
                cur.right = cur.left
            cur.left = p
            p.parent = cur
            if p.left is cur:
                p.left = None
            else:
                p.right = None
            cur, p = p, gp
        leaf.parent = None
        return leaf

for arr, lv, expected in examples:
    root = build_tree(arr)
    nr = Solution().flipBinaryTree(root, find(root, lv))
    print("leaf", lv, "->", serialize(nr), "| parent ok:", parents_ok(nr))
    assert serialize(nr) == expected and parents_ok(nr)

leaf 7 -> [7, 2, None, 5, 4, 3, 6, None, None, None, 1, None, None, 0, 8] | parent ok: True
leaf 0 -> [0, 1, None, 3, 8, 5, None, None, None, 6, 2, None, None, 7, 4] | parent ok: True


## 10. Vì sao thuật toán đúng

Gọi đường đi lá → gốc là `v₀ = leaf, v₁, …, v_k = root`.

**Invariant (trước vòng lặp thứ `i`, với `cur = vᵢ`, `p = vᵢ₊₁`, `i < k`):**
1. Với mọi `j < i`: `vⱼ.left = vⱼ₊₁`, `vⱼ₊₁.parent = vⱼ`, và con còn lại ban đầu của `vⱼ` (nếu có) là `vⱼ.right`.
2. `vᵢ` **không** còn con trỏ xuống `vᵢ₋₁` (đã bị cắt ở vòng `i−1`; với `i = 0` lá không có con).
3. `vᵢ.parent` vẫn là `vᵢ₊₁` cũ và `vᵢ₊₁.parent` vẫn là `vᵢ₊₂` cũ (vì chưa vòng nào chạm tới hai con trỏ này) ⇒ lưu `gp` trước là đúng.

**Bảo toàn:** trong vòng `i`, ta chuyển con còn lại (nếu ở trái) sang phải — không đè vì theo (2) bên phải đang trống hoặc chính là con còn lại; gán `vᵢ.left = vᵢ₊₁`, `vᵢ₊₁.parent = vᵢ`; cắt `vᵢ₊₁ → vᵢ` ⇒ (1), (2) đúng cho `i+1`.

**Kết thúc** khi `cur = root`: gốc cũ đã bị cắt con trên đường đi ở vòng cuối, giữ nguyên con còn lại; mọi nút **ngoài** đường đi không bị chạm nên cây con của chúng nguyên vẹn. Cuối cùng `leaf.parent = None`. Mỗi cạnh trên đường đi được đảo chiều đúng một lần, tập cạnh vô hướng không đổi ⇒ kết quả là cây hợp lệ với gốc `leaf`, đúng quy tắc của đề.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút (`2 ≤ n ≤ 100`), `h` là độ sâu của `leaf` (`h ≤ n − 1`).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. DFS + dựng lại parent | `O(n·h)` (có thể `O(n)`) | `O(n)` | chạm cả cây |
| 2. Lặp bằng parent | `O(h)` | `O(1)` | **nộp bài** |
| 3. Đệ quy | `O(h)` | `O(h)` stack | `h ≤ 99` nên an toàn |

Trường hợp tệ nhất `h = n − 1` (cây dạng "dây chuyền") ⇒ `O(n)`. Trung bình với cây ngẫu nhiên, `h` cỡ `O(log n)`–`O(√n)`, nên Cách 2 thường chỉ chạm rất ít nút.

## 12. Bộ test

**Validator độc lập** (không dùng lại code lời giải) kiểm tra kết quả có đúng quy tắc của đề:
1. Gốc mới là `leaf`, `parents_ok` đúng.
2. Với mỗi nút `vᵢ` trên đường đi (trừ gốc cũ): `vᵢ.left` là cha cũ; `vᵢ.right` là con còn lại ban đầu (con không nằm trên đường đi, dù ban đầu nó ở trái hay phải).
3. Gốc cũ: giữ đúng con còn lại ở **đúng phía cũ** của nó.
4. Mọi nút ngoài đường đi: `left`, `right` y như ban đầu.

Ta chụp "ảnh" cây (dict theo giá trị) trước khi chạy lời giải.

In [15]:
def snapshot(root):
    """dict: val -> (val trái, val phải, val cha) của cây ban đầu."""
    snap, st = {}, [root]
    while st:
        x = st.pop()
        snap[x.val] = (x.left.val if x.left else None, x.right.val if x.right else None,
                       x.parent.val if x.parent else None)
        for c in (x.left, x.right):
            if c: st.append(c)
    return snap

def valid_reroot(snap, old_root_val, leaf_val, new_root):
    if new_root is None or new_root.val != leaf_val or not parents_ok(new_root):
        return False
    after = snapshot(new_root)
    if set(after) != set(snap):
        return False
    path = [leaf_val]                       # đường đi theo cây ban đầu
    while snap[path[-1]][2] is not None:
        path.append(snap[path[-1]][2])
    on_path = set(path)
    for i, v in enumerate(path):
        L, R, _ = snap[v]
        below = path[i - 1] if i > 0 else None
        other = [c for c in (L, R) if c is not None and c != below]
        if v == old_root_val:
            exp = (L if L != below else None, R if R != below else None)
        else:
            exp = (path[i + 1], other[0] if other else None)
        if after[v][:2] != exp:
            return False
    for v, (L, R, _) in snap.items():
        if v not in on_path and after[v][:2] != (L, R):
            return False
    return True

# validator phải chấp nhận đáp án đề và từ chối một cây sai
root = build_tree(EX_TREE); snap = snapshot(root)
good = build_tree(examples[0][2])
print("validator với đáp án đề:", valid_reroot(snap, 3, 7, good))
bad = build_tree([7, 2, None, 5, 4, 6, 3, None, None, None, None, None, 1, 0, 8])   # đổi chỗ 3 và 6
print("validator với cây sai:", valid_reroot(snap, 3, 7, bad))
assert valid_reroot(snap, 3, 7, good) and not valid_reroot(snap, 3, 7, bad)

validator với đáp án đề: True
validator với cây sai: False


In [16]:
SOLS = [("baseline", flip_baseline),
        ("iter", SolutionIter().flipBinaryTree),
        ("rec", SolutionRec().flipBinaryTree),
        ("submit", Solution().flipBinaryTree)]

# 1) ví dụ đề, mọi cách
for arr, lv, expected in examples:
    for name, f in SOLS:
        root = build_tree(arr)
        nr = f(root, find(root, lv))
        assert serialize(nr) == expected and parents_ok(nr), name

# 2) edge case (tự dựng — đã kiểm bằng validator; đối chiếu lại đề gốc nếu cần)
edge = [
    ([1, 2], 2, [2, 1]),                       # 2 nút, lá là con trái
    ([1, None, 2], 2, [2, 1]),                 # 2 nút, lá là con phải
    ([1, 2, 3], 3, [3, 1, None, 2]),           # gốc cũ giữ con còn lại ở phía cũ (trái)
    ([1, 2, 3], 2, [2, 1, None, None, 3]),     # gốc cũ giữ con phải
    ([1, 2, None, 3, None, 4], 4, [4, 3, None, 2, None, 1]),   # dây chuyền toàn trái
    ([1, 2, None, 3, 5, None, 4], 4, [4, 3, None, 2, None, 1, 5]),
]
for arr, lv, expected in edge:
    for name, f in SOLS:
        root = build_tree(arr); snap = snapshot(root)
        nr = f(root, find(root, lv))
        assert serialize(nr) == expected, (name, arr, lv, serialize(nr))
        assert valid_reroot(snap, arr[0], lv, nr)
print("ví dụ đề + edge case OK")

ví dụ đề + edge case OK


In [17]:
# 3) ngẫu nhiên: mọi cách, mọi lá, cây 2..100 nút (seed cố định) + so sánh chéo
import contextlib, io
rnd = random.Random(2024)
tests = 0
for _ in range(300):
    n = rnd.randint(2, 100)
    seed = rnd.randrange(10**9)
    base = random_tree(n, random.Random(seed))
    for lf in leaves(base)[:5]:
        outs = []
        for name, f in SOLS:
            root = random_tree(n, random.Random(seed))       # dựng lại cây y hệt
            snap = snapshot(root)
            leaf = find(root, lf.val)
            nr = f(root, leaf)
            assert valid_reroot(snap, root.val, lf.val, nr), name
            outs.append(serialize(nr))
        assert all(o == outs[0] for o in outs)
        tests += 1
print("số test ngẫu nhiên:", tests, "OK")

số test ngẫu nhiên: 1454 OK


In [18]:
# 4) cây dây chuyền sâu nhất (n = 100) — kiểm tra đệ quy không vượt giới hạn và đường đi dài
for zig in (False, True):
    root = Node(0); x = root
    for v in range(1, 100):
        c = Node(v); c.parent = x
        if zig and v % 2: x.right = c
        else: x.left = c
        x = c
    snap = snapshot(root)
    nr = SolutionRec().flipBinaryTree(root, x)
    assert valid_reroot(snap, 0, 99, nr)
print("dây chuyền 100 nút OK")

dây chuyền 100 nút OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên cập nhật `parent`

Chỉ sửa `left/right` mà bỏ qua `parent` ⇒ cấu trúc nhìn đúng khi serialize nhưng LeetCode vẫn báo sai (vì họ kiểm tra `parent`).

In [19]:
def wrong_no_parent(root, leaf):
    cur, p = leaf, leaf.parent
    while cur is not root:
        gp = p.parent
        if cur.left:
            cur.right = cur.left
        cur.left = p
        if p.left is cur: p.left = None
        else: p.right = None
        cur, p = p, gp            # SAI: không gán p.parent = cur, không reset leaf.parent
    return leaf

root = build_tree(EX_TREE)
nr = wrong_no_parent(root, find(root, 7))
print("serialize:", serialize(nr), "== đáp án:", serialize(nr) == examples[0][2])
print("parent ok?", parents_ok(nr), "| leaf.parent =", nr.parent)

serialize: [7, 2, None, 5, 4, 3, 6, None, None, None, 1, None, None, 0, 8] == đáp án: True
parent ok? False | leaf.parent = Node(2)


**Vì sao sai:** serialize chỉ nhìn `left/right` nên trông đúng, nhưng `7.parent` vẫn là `2`, `2.parent` vẫn là `5`… **Sửa:** `p.parent = cur` trong vòng lặp và `leaf.parent = None` ở cuối.

### Lỗi 2 — Không lưu `gp` trước khi ghi đè `p.parent`

In [20]:
def wrong_no_gp(root, leaf):
    cur = leaf
    steps = 0
    while cur is not root and steps < 10:     # giới hạn để không lặp vô hạn trong demo
        p = cur.parent
        if cur.left:
            cur.right = cur.left
        cur.left = p
        p.parent = cur                         # ghi đè ...
        if p.left is cur: p.left = None
        else: p.right = None
        cur = p.parent                         # SAI: p.parent giờ là cur -> quay lại chính cur
        steps += 1
    leaf.parent = None
    return leaf

root = build_tree(EX_TREE)
nr = wrong_no_gp(root, find(root, 7))
snap = snapshot(build_tree(EX_TREE))
print("số bước đã chạy:", 10, "(bị kẹt vòng lặp) | hợp lệ?", valid_reroot(snap, 3, 7, nr))

số bước đã chạy: 10 (bị kẹt vòng lặp) | hợp lệ? False


**Vì sao sai:** sau `p.parent = cur`, lệnh `cur = p.parent` đưa ta **trở lại** `cur` — vòng lặp không tiến lên được (ở đây ta chặn bằng `steps < 10`; trên LeetCode sẽ lặp vô hạn ⇒ TLE). **Sửa:** `gp = p.parent` **trước** khi đổi bất kỳ con trỏ nào, cuối vòng `cur, p = p, gp`.

### Lỗi 3 — Gán `cur.left = p` trước khi chuyển con trái sang phải

In [21]:
def wrong_order(root, leaf):
    cur, p = leaf, leaf.parent
    while cur is not root:
        gp = p.parent
        cur.left = p                    # SAI: ghi đè con trái trước ...
        if cur.left:                    # ... nên giờ "con trái" chính là p
            cur.right = cur.left
        p.parent = cur
        if p.left is cur: p.left = None
        else: p.right = None
        cur, p = p, gp
    leaf.parent = None
    return leaf

root = build_tree(EX_TREE); snap = snapshot(root)
nr = wrong_order(root, find(root, 7))
print("hợp lệ?", valid_reroot(snap, 3, 7, nr))
print("parent ok?", parents_ok(nr))
out = serialize(nr)
print("serialize:", out)
print("nút 6 còn trong cây?", 6 in out)

hợp lệ? False
parent ok? False
serialize: [7, 2, 2, 5, 5, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, None, 1, None, 1, None, 1, None, 1, None, 1, None, 1, None, 1, None, 1, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8]
nút 6 còn trong cây? False


**Vì sao sai:** con trái cũ (ví dụ `6` của nút `5`) bị **mất** vì ta ghi đè `cur.left` trước khi chép nó sang phải; đồng thời `cur.right` lại trỏ tới `p` ⇒ `p` có hai cha. **Sửa:** chép `cur.right = cur.left` **trước**, rồi mới `cur.left = p`.

### Lỗi 4 — Xử lý cả gốc cũ như một nút bình thường

Nếu vòng lặp chạy cả khi `cur is root` thì `p = root.parent = None` ⇒ `None.parent` gây lỗi; hoặc nếu chặn lỗi bằng `if p:` mà vẫn "chuyển trái sang phải" ở gốc cũ thì gốc cũ bị đổi phía con trái → **sai quy tắc** (đề nói *không tính root*).

In [22]:
def wrong_touch_root(root, leaf):
    cur, p = leaf, leaf.parent
    while cur is not None:
        gp = p.parent if p else None
        if cur.left:
            cur.right = cur.left        # SAI ở gốc cũ: con trái của gốc bị chuyển sang phải
            cur.left = None
        if p:
            cur.left = p
            p.parent = cur
            if p.left is cur: p.left = None
            else: p.right = None
        cur, p = p, gp
    leaf.parent = None
    return leaf

root = build_tree([1, 2, 3]); snap = snapshot(root)
nr = wrong_touch_root(root, find(root, 3))
print("sai :", serialize(nr), "hợp lệ?", valid_reroot(snap, 1, 3, nr))
root = build_tree([1, 2, 3])
print("đúng:", serialize(Solution().flipBinaryTree(root, find(root, 3))))

sai : [3, 1, None, None, 2] hợp lệ? False
đúng: [3, 1, None, 2]


**Vì sao sai:** với `[1,2,3]`, `leaf = 3`, gốc cũ `1` phải **giữ** con trái `2` (đáp án `[3,1,null,2]`), nhưng code sai chuyển `2` sang phải. **Sửa:** dừng vòng lặp khi `cur is root`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đổi gốc", "lật cây", "reroot" ⇒ thực chất là **đảo chiều các cạnh trên một đường đi**; phần còn lại của cây không đổi.
- Có con trỏ `parent` ⇒ đi **từ dưới lên** là tự nhiên nhất; giống **đảo ngược danh sách liên kết** (LC 206): lưu `next` (ở đây là `gp`) trước khi ghi đè.
- Mỗi bước cập nhật **3 loại con trỏ**: con của `cur`, `parent` của `p`, con của `p`. Viết ra giấy 4 dòng trước khi code.
- So sánh nút bằng `is`, không bằng giá trị (dù đề cho giá trị khác nhau, `is` vẫn an toàn và rõ nghĩa hơn).
- Khi debug: viết `parents_ok` và `print_tree` có in `parent` — lỗi con trỏ `parent` không hiện ra khi chỉ serialize.

## 15. Biến thể của bài

1. **LC 206 — Reverse Linked List:** cùng kỹ thuật "lưu next, đảo con trỏ, tiến lên".
2. **LC 156 — Binary Tree Upside Down (Premium):** lật cây theo nhánh trái, cũng đảo chiều cạnh trên một đường đi.
3. **LC 1650 — Lowest Common Ancestor of a Binary Tree III:** dùng con trỏ `parent` đi lên.
4. **Đổi gốc cây tổng quát** (cây biểu diễn bằng mảng cha): đảo mảng cha trên đường đi — bài tập 5.
5. **Rerooting DP** (ví dụ LC 834 — Sum of Distances in Tree): tính đáp án cho mọi gốc bằng cách "dời gốc" qua từng cạnh.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS + BFS dựng lại parent | tìm đường đi không cần parent, lật, gán lại parent toàn cây | `O(n·h)`/`O(n)` | `O(n)` | khi không tin con trỏ parent, để kiểm chứng |
| 2. Lặp bằng parent | lưu `gp`, lật, tiến lên | `O(h)` | `O(1)` | **nộp bài** |
| 3. Đệ quy `attach` | lật cạnh rồi đệ quy lên cha cũ | `O(h)` | `O(h)` | khi thích tư duy đệ quy; `h` nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Serialize kết quả của (a) `root = [1,2,3]`, `leaf = 3`; (b) `root = [1,2,3,4]`, `leaf = 4`; (c) `root = [1,2,3,null,5]`, `leaf = 5`.

**Bài 2 (tính tay):** Với cây ví dụ và `leaf = 4`, liệt kê đường đi lá → gốc và cho biết sau khi đổi gốc, mỗi nút trên đường đi có `left`, `right` là gì.

**Bài 3 (điền TODO):** Hoàn thiện `path_to_root(leaf)` trả danh sách **nút** từ `leaf` lên gốc (dùng `parent`).

**Bài 4 (tự viết lại):** Viết lại lời giải theo kiểu "lưu đường đi trước": lấy `path = path_to_root(leaf)`, rồi lật cạnh lần lượt bằng chỉ số `k` (không dùng biến `gp`).

**Bài 5 (mini challenge):** Cây tổng quát `n` nút cho bằng mảng cha `par` (`par[root] = -1`). Viết `reroot_parent_array(par, r)` trả mảng cha mới khi gốc là `r` (đảo các cạnh trên đường đi `r → gốc cũ`).

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def path_to_root(leaf):
    path = []
    # TODO: đi lên bằng parent cho tới None
    return path

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def flip_by_path(root, leaf):
    # TODO
    pass

def reroot_parent_array(par, r):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tìm đường đi lá → gốc. Mức 2: gốc cũ giữ con còn lại ở phía cũ. Mức 3: (a) `[3,1,null,2]`; (b) đường đi `4→2→1`: `[4,2,null,1,null,null,3]`; (c) đường đi `5→2→1`: `[5,2,null,1,null,null,3]`.

**Bài 2.** Mức 1: đường đi `4 → 2 → 5 → 3`. Mức 2: `4.left = 2`; `2` có con còn lại `7` ở trái ⇒ chuyển sang phải. Mức 3: `4:(2,—)`, `2:(5,7)`, `5:(3,6)`, `3:(—,1)`.

**Bài 3.** Mức 1: `while node:`. Mức 2: `path.append(node); node = node.parent`. Mức 3: kiểm tra `path[-1] is root`.

**Bài 4.** Mức 1: với `k` từ `0` tới `len(path)−2`: `cur = path[k]`, `p = path[k+1]`. Mức 2: cắt cạnh `p → cur`, chuyển trái sang phải, `cur.left = p`, `p.parent = cur`. Mức 3: cuối cùng `leaf.parent = None`.

**Bài 5.** Mức 1: lấy đường đi `r = u₀, u₁ = par[u₀], …, gốc cũ`. Mức 2: đặt `new[u_{i+1}] = u_i` cho mọi `i`, `new[r] = −1`. Mức 3: kiểm tra bằng BFS từ `r` trên đồ thị vô hướng.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
b1 = []
for arr, lv in (([1, 2, 3], 3), ([1, 2, 3, 4], 4), ([1, 2, 3, None, 5], 5)):
    root = build_tree(arr)
    b1.append(serialize(Solution().flipBinaryTree(root, find(root, lv))))
print("Bài 1:", b1)
assert b1 == [[3, 1, None, 2], [4, 2, None, 1, None, None, 3], [5, 2, None, 1, None, None, 3]]

# Bài 2
root = build_tree(EX_TREE)
leaf = find(root, 4)
print("đường đi:", path_up(leaf))
nr = Solution().flipBinaryTree(root, leaf)
info = {v: (find(nr, v).left.val if find(nr, v).left else None,
            find(nr, v).right.val if find(nr, v).right else None) for v in (4, 2, 5, 3)}
print("Bài 2:", info)
assert info == {4: (2, None), 2: (5, 7), 5: (3, 6), 3: (None, 1)}

Bài 1: [[3, 1, None, 2], [4, 2, None, 1, None, None, 3], [5, 2, None, 1, None, None, 3]]
đường đi: [4, 2, 5, 3]
Bài 2: {4: (2, None), 2: (5, 7), 5: (3, 6), 3: (None, 1)}


In [26]:
# Bài 3, 4
def path_to_root_sol(leaf):
    path = []
    node = leaf
    while node is not None:
        path.append(node)
        node = node.parent
    return path

def flip_by_path_sol(root, leaf):
    path = path_to_root_sol(leaf)
    for k in range(len(path) - 1):
        cur, p = path[k], path[k + 1]
        if p.left is cur: p.left = None
        else: p.right = None
        if cur.left is not None:
            cur.right = cur.left
        cur.left = p
        p.parent = cur
    leaf.parent = None
    return leaf

root = build_tree(EX_TREE)
assert [x.val for x in path_to_root_sol(find(root, 7))] == [7, 2, 5, 3]
rnd = random.Random(77)
for _ in range(300):
    n = rnd.randint(2, 100); seed = rnd.randrange(10**9)
    t = random_tree(n, random.Random(seed))
    lf = rnd.choice(leaves(t)).val
    r1 = random_tree(n, random.Random(seed)); s1 = snapshot(r1)
    a = flip_by_path_sol(r1, find(r1, lf))
    r2 = random_tree(n, random.Random(seed))
    b = Solution().flipBinaryTree(r2, find(r2, lf))
    assert valid_reroot(s1, t.val, lf, a) and serialize(a) == serialize(b)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [27]:
# Bài 5
def reroot_parent_array_sol(par, r):
    new = par[:]
    prev, u = -1, r
    while u != -1:
        nxt = par[u]
        new[u] = prev
        prev, u = u, nxt
    return new

def reroot_brute(par, r):
    n = len(par)
    adj = [[] for _ in range(n)]
    for v, p in enumerate(par):
        if p != -1:
            adj[v].append(p); adj[p].append(v)
    new = [None] * n; new[r] = -1
    q = deque([r])
    while q:
        x = q.popleft()
        for y in adj[x]:
            if new[y] is None:
                new[y] = x; q.append(y)
    return new

print("ví dụ:", reroot_parent_array_sol([-1, 0, 0, 1, 1], 3))
rnd = random.Random(5)
for _ in range(1000):
    n = rnd.randint(1, 60)
    perm = list(range(n)); rnd.shuffle(perm)
    par = [-1] * n
    for i in range(1, n):
        par[perm[i]] = perm[rnd.randrange(i)]
    r = rnd.randrange(n)
    assert reroot_parent_array_sol(par, r) == reroot_brute(par, r)
print("Bài 5 OK")

ví dụ: [1, 3, 0, -1, 1]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đảo chiều cạnh trên đường đi lá → gốc" (có parent):**

```python
cur, p = leaf, leaf.parent
while cur is not root:
    gp = p.parent                  # 1) lưu ông TRƯỚC
    if cur.left:                   # 2) con trái còn lại -> phải
        cur.right = cur.left
    cur.left = p                   # 3) cha cũ -> con trái
    p.parent = cur                 # 4) cập nhật parent
    if p.left is cur: p.left = None    # 5) cắt cạnh cha -> cur
    else:             p.right = None
    cur, p = p, gp                 # 6) tiến lên
leaf.parent = None                 # 7) gốc mới không có cha
return leaf
```

**Checklist:**
- [ ] Lưu `gp = p.parent` trước khi ghi `p.parent = cur`?
- [ ] Chép `cur.left` sang `cur.right` **trước** khi gán `cur.left = p`?
- [ ] Cắt con trỏ từ cha cũ xuống `cur` (so sánh bằng `is`)?
- [ ] Không xử lý gốc cũ (dừng khi `cur is root`)?
- [ ] `leaf.parent = None` ở cuối?
- [ ] Đã kiểm tra `parent` của mọi nút (không chỉ serialize) trên nhiều cây ngẫu nhiên?